# Model training notebook

# Import necessary libraries and functions from the kso2 package

In [ ]:
# These functions handle project creation, data management, model training, and serving.
from kso import (
    ProjectManager,
    TrainingManager,
    MLflowServerManager,
    InferenceManager,
)

## Create a new project instance.

In [ ]:
# This initializes the project structure, creating necessary folders and configuration files.
proj = ProjectManager()
project = proj.load_project(yaml_path="projects/test_project/test_project.project.yaml")

## Configure the model to be used for training.

In [ ]:
# 'yolov8n.pt' is a nano version of YOLOv8, which is small and fast, making it good for quick tests.
# You can specify other YOLO versions or provide a path to a local model file.
# See available models here: https://docs.ultralytics.com/models/
proj.add_model(
    project=project,
    model_path="yolo26x.pt",
    model_name="latest model",
)

## Start the training process.

In [ ]:
# 'epochs' determines the number of training passes over the dataset.
# We use 1 epoch here for a quick demonstration. For real training, you would typically use 50-100+ epochs.
# add link to yolo model parameters
train = TrainingManager()
train.train_model(project=project, epochs=1)

In [ ]:
# retrieve registered models
serve = MLflowServerManager()
listed_models = serve.get_registered_models(project)
listed_models

In [ ]:
# `model_name`, `version`, and `split` are optional for selecting a model from MLflow registered models; if not provided, the latest trained model will be used by default.
inference = InferenceManager()
inference.model_validation(
    project=project,
    model_name="best model",
    version=1,
    split="val",
    data_path="kso/projectss/data/test_yolo.yaml",
)  # data_path= "" for path to custom data

## Start the MLflow server to track experiments and visualize results.

In [ ]:
# The server runs in the background and logs output to 'mlflow.log'.
# You can access the MLflow UI in your browser (usually at http://127.0.0.1:8080).
serve = MLflowServerManager()
serve.start_mlflow_server(project=project)

## Export the experiment artifacts.

In [ ]:
# This saves the trained model, configuration, and metadata.
# The exported model can be reused later or deployed to a production environment.
# NOTE: The MLflow server needs to be running before exporting.
train.export_experiment(project)

## Import the experiment artifacts.

In [ ]:
# Imports a previously exported MLflow experiment into the current MLflow tracking server.
# This restores runs, metrics, parameters, artifacts, and experiment structure from the export directory.
train.import_experiment(project, input_dir="projectss/test_project/output")

## Stop the MLflow server.

In [ ]:
# Use this to shut down the background process when you are finished with the notebook.
serve.stop_mlflow_server()